
# Microstructure Mining + Backtest + Meta-Labeling Notebook

Notebook ini dirancang untuk data **XAUUSD M1 ~5 tahun** dengan fokus pada:

- data quality check
- microstructure proxy feature engineering
- event mining
- backtest execution-aware
- baseline statistical validation
- meta-labeling dataset
- model baseline (Logistic Regression + optional Gradient Boosting)

Asumsi utama:
- Entry: next bar open
- SL: structural swing high/low
- TP: 2R
- Timeout: 50 bar
- Spread: 0
- Intrabar tie: SL-first


In [ ]:

import json
import math
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:

@dataclass
class ResearchConfig:
    csv_path: str = "PATH/TO/XAUUSD_M1.csv"
    out_dir: str = "./outputs_microstructure"
    timezone: str = "UTC"
    split_train: float = 0.60
    split_val: float = 0.20
    split_test: float = 0.20

    # Event parameters
    rolling_n: int = 20
    inside_cluster_n: int = 3
    compression_window: int = 20
    momentum_body_ratio: float = 0.70
    rejection_wick_ratio: float = 0.50
    compression_quantile: float = 0.20
    expansion_body_quantile: float = 0.80

    # Structural SL
    swing_lookback: int = 10
    min_r_distance: float = 0.05

    # Execution
    rr_tp: float = 2.0
    timeout_bars: int = 50
    spread_points: float = 0.0
    intrabar_tie_priority: str = "SL_FIRST"

    # Meta-labeling
    prob_threshold: float = 0.55
    random_state: int = 42

CFG = ResearchConfig()
OUT_DIR = Path(CFG.out_dir)
OUT_DIR.mkdir(parents=True, exist_ok=True)
CFG


## 1. Loader dan quality checks

In [ ]:

def load_ohlcv(path: str) -> pd.DataFrame:
    names = ["date", "time", "open", "high", "low", "close", "volume"]
    df = pd.read_csv(path, names=names, skiprows=1)
    ts = pd.to_datetime(df["date"] + " " + df["time"], format="%Y.%m.%d %H:%M", utc=True)
    df = df.drop(columns=["date", "time"])
    df.index = pd.DatetimeIndex(ts, name="timestamp")
    df = df.sort_index()
    df = df[~df.index.duplicated(keep="first")]
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def quality_report(df: pd.DataFrame) -> Dict[str, object]:
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq="1min", tz="UTC")
    missing = full_idx.difference(df.index)
    report = {
        "rows": int(len(df)),
        "start": str(df.index.min()),
        "end": str(df.index.max()),
        "duplicate_index": int(df.index.duplicated().sum()),
        "missing_bars": int(len(missing)),
        "nan_cells": int(df.isna().sum().sum()),
        "non_positive_open": int((df["open"] <= 0).sum()),
        "non_positive_high": int((df["high"] <= 0).sum()),
        "non_positive_low": int((df["low"] <= 0).sum()),
        "non_positive_close": int((df["close"] <= 0).sum()),
    }
    ret_abs = df["close"].pct_change().abs()
    report["outlier_abs_return_gt_1pct"] = int((ret_abs > 0.01).sum())
    return report

def print_quality_report(report: Dict[str, object]) -> None:
    print(json.dumps(report, indent=2))


In [ ]:

# Uncomment setelah CFG.csv_path diisi benar
# df_raw = load_ohlcv(CFG.csv_path)
# report = quality_report(df_raw)
# print_quality_report(report)
# df_raw.head()


## 2. Feature engineering microstructure proxy

In [ ]:

def rolling_streak(sign_series: pd.Series) -> pd.Series:
    out = np.zeros(len(sign_series), dtype=int)
    prev = 0
    count = 0
    values = sign_series.fillna(0).astype(int).values
    for i, v in enumerate(values):
        if v == 0:
            count = 0
        elif v == prev:
            count += 1
        else:
            count = 1
        out[i] = count * v
        prev = v
    return pd.Series(out, index=sign_series.index)

def add_session_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["hour"] = out.index.hour
    conditions = [
        out["hour"].between(0, 7),
        out["hour"].between(8, 15),
        out["hour"].between(16, 23),
    ]
    choices = ["Asia", "London", "NY"]
    out["session"] = np.select(conditions, choices, default="Other")
    return out

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["range"] = out["high"] - out["low"]
    out["body"] = out["close"] - out["open"]
    out["body_abs"] = out["body"].abs()
    out["upper_wick"] = out["high"] - out[["open", "close"]].max(axis=1)
    out["lower_wick"] = out[["open", "close"]].min(axis=1) - out["low"]
    out["body_ratio"] = np.where(out["range"] > 0, out["body_abs"] / out["range"], np.nan)
    out["close_position"] = np.where(out["range"] > 0, (out["close"] - out["low"]) / out["range"], np.nan)

    for n in [1, 3, 5, 10, 20]:
        out[f"ret_{n}"] = out["close"].pct_change(n)
        out[f"range_mean_{n}"] = out["range"].rolling(n).mean()
        out[f"vol_std_{n}"] = out["close"].pct_change().rolling(n).std()

    prev_close = out["close"].shift(1)
    tr = pd.concat([
        out["high"] - out["low"],
        (out["high"] - prev_close).abs(),
        (out["low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    out["tr"] = tr
    out["atr_14"] = tr.rolling(14).mean()
    out["atr_50"] = tr.rolling(50).mean()
    out["compression_ratio_20"] = out["range_mean_5"] / out["range_mean_20"]

    for n in [10, 20, 50]:
        out[f"roll_high_{n}"] = out["high"].shift(1).rolling(n).max()
        out[f"roll_low_{n}"] = out["low"].shift(1).rolling(n).min()
        out[f"dist_high_{n}"] = out["close"] - out[f"roll_high_{n}"]
        out[f"dist_low_{n}"] = out["close"] - out[f"roll_low_{n}"]

    for span in [20, 50]:
        out[f"ema_{span}"] = out["close"].ewm(span=span, adjust=False).mean()
        out[f"ema_slope_{span}"] = out[f"ema_{span}"].diff(3)
        out[f"dist_ema_{span}"] = out["close"] - out[f"ema_{span}"]

    bb_mid = out["close"].rolling(20).mean()
    bb_std = out["close"].rolling(20).std()
    out["bb_mid"] = bb_mid
    out["bb_upper"] = bb_mid + 2 * bb_std
    out["bb_lower"] = bb_mid - 2 * bb_std
    out["bb_width"] = out["bb_upper"] - out["bb_lower"]
    out["bb_width_atr"] = out["bb_width"] / out["atr_14"]

    candle_sign = np.sign(out["close"].diff())
    out["streak"] = rolling_streak(candle_sign)
    net_move = out["close"].diff(10).abs()
    gross_move = out["close"].diff().abs().rolling(10).sum()
    out["directional_eff_10"] = np.where(gross_move > 0, net_move / gross_move, np.nan)

    out["inside_bar"] = (
        (out["high"] <= out["high"].shift(1)) &
        (out["low"] >= out["low"].shift(1))
    ).astype(int)
    out["inside_cluster"] = out["inside_bar"].rolling(CFG.inside_cluster_n).sum()

    out = add_session_features(out)
    return out


In [ ]:

# df = engineer_features(df_raw)
# df.tail()


## 3. Definisi event candidate

In [ ]:

def build_events(df: pd.DataFrame, cfg: ResearchConfig) -> pd.DataFrame:
    out = df.copy()
    prev_high = out["high"].shift(1)
    prev_low = out["low"].shift(1)

    out["evt_momentum_long"] = (
        (out["body_ratio"] >= cfg.momentum_body_ratio) &
        (out["close"] > out["open"]) &
        (out["high"] > prev_high)
    )
    out["evt_momentum_short"] = (
        (out["body_ratio"] >= cfg.momentum_body_ratio) &
        (out["close"] < out["open"]) &
        (out["low"] < prev_low)
    )

    out["evt_sweep_reclaim_long"] = (
        (out["low"] < out["roll_low_20"]) &
        (out["close"] > out["roll_low_20"]) &
        (out["close"] > out["open"])
    )
    out["evt_sweep_reclaim_short"] = (
        (out["high"] > out["roll_high_20"]) &
        (out["close"] < out["roll_high_20"]) &
        (out["close"] < out["open"])
    )

    comp_thresh = out["compression_ratio_20"].rolling(200).quantile(cfg.compression_quantile)
    body_thresh = out["body_abs"].rolling(200).quantile(cfg.expansion_body_quantile)
    out["evt_comp_exp_long"] = (
        (out["compression_ratio_20"] <= comp_thresh) &
        (out["body_abs"] >= body_thresh) &
        (out["close"] > out["open"])
    )
    out["evt_comp_exp_short"] = (
        (out["compression_ratio_20"] <= comp_thresh) &
        (out["body_abs"] >= body_thresh) &
        (out["close"] < out["open"])
    )

    out["evt_inside_break_long"] = (
        (out["inside_cluster"].shift(1) >= cfg.inside_cluster_n) &
        (out["close"] > out["high"].shift(1))
    )
    out["evt_inside_break_short"] = (
        (out["inside_cluster"].shift(1) >= cfg.inside_cluster_n) &
        (out["close"] < out["low"].shift(1))
    )

    out["evt_rejection_long"] = (
        (out["lower_wick"] >= cfg.rejection_wick_ratio * out["range"]) &
        (out["close"] > out["open"])
    )
    out["evt_rejection_short"] = (
        (out["upper_wick"] >= cfg.rejection_wick_ratio * out["range"]) &
        (out["close"] < out["open"])
    )
    return out

EVENT_MAP = {
    "momentum_long": ("evt_momentum_long", 1),
    "momentum_short": ("evt_momentum_short", -1),
    "sweep_reclaim_long": ("evt_sweep_reclaim_long", 1),
    "sweep_reclaim_short": ("evt_sweep_reclaim_short", -1),
    "comp_exp_long": ("evt_comp_exp_long", 1),
    "comp_exp_short": ("evt_comp_exp_short", -1),
    "inside_break_long": ("evt_inside_break_long", 1),
    "inside_break_short": ("evt_inside_break_short", -1),
    "rejection_long": ("evt_rejection_long", 1),
    "rejection_short": ("evt_rejection_short", -1),
}


## 4. Structural swing SL + triple barrier backtest

In [ ]:

def structural_sl(df: pd.DataFrame, idx: int, direction: int, lookback: int) -> Optional[float]:
    if idx <= lookback:
        return None
    window = df.iloc[idx - lookback: idx]
    return float(window["low"].min()) if direction == 1 else float(window["high"].max())

def simulate_trade(
    df: pd.DataFrame,
    signal_idx: int,
    direction: int,
    event_name: str,
    cfg: ResearchConfig,
) -> Optional[Dict[str, object]]:
    entry_idx = signal_idx + 1
    if entry_idx >= len(df) - 1:
        return None

    entry_ts = df.index[entry_idx]
    entry_price = float(df.iloc[entry_idx]["open"])
    sl_price = structural_sl(df, entry_idx, direction, cfg.swing_lookback)
    if sl_price is None:
        return None

    risk = (entry_price - sl_price) if direction == 1 else (sl_price - entry_price)
    if not np.isfinite(risk) or risk <= cfg.min_r_distance:
        return None

    tp_price = entry_price + cfg.rr_tp * risk if direction == 1 else entry_price - cfg.rr_tp * risk

    exit_reason = "timeout"
    exit_idx = min(entry_idx + cfg.timeout_bars, len(df) - 1)
    exit_ts = df.index[exit_idx]
    exit_price = float(df.iloc[exit_idx]["close"])
    r_multiple = ((exit_price - entry_price) / risk) if direction == 1 else ((entry_price - exit_price) / risk)

    for i in range(entry_idx, min(entry_idx + cfg.timeout_bars + 1, len(df))):
        row = df.iloc[i]
        bar_open = float(row["open"])
        bar_high = float(row["high"])
        bar_low = float(row["low"])
        ts = df.index[i]

        if direction == 1 and bar_open <= sl_price:
            exit_reason, exit_idx, exit_ts, exit_price, r_multiple = "gap_sl", i, ts, bar_open, (bar_open - entry_price) / risk
            break
        if direction == -1 and bar_open >= sl_price:
            exit_reason, exit_idx, exit_ts, exit_price, r_multiple = "gap_sl", i, ts, bar_open, (entry_price - bar_open) / risk
            break

        sl_hit = bar_low <= sl_price if direction == 1 else bar_high >= sl_price
        tp_hit = bar_high >= tp_price if direction == 1 else bar_low <= tp_price

        if sl_hit and tp_hit:
            exit_reason, exit_idx, exit_ts, exit_price, r_multiple = "sl_tie", i, ts, sl_price, -1.0
            break
        elif sl_hit:
            exit_reason, exit_idx, exit_ts, exit_price, r_multiple = "sl", i, ts, sl_price, -1.0
            break
        elif tp_hit:
            exit_reason, exit_idx, exit_ts, exit_price, r_multiple = "tp", i, ts, tp_price, cfg.rr_tp
            break

    label = 1 if exit_reason == "tp" else 0

    return {
        "event_name": event_name,
        "direction": direction,
        "signal_ts": str(df.index[signal_idx]),
        "entry_ts": str(entry_ts),
        "exit_ts": str(exit_ts),
        "entry_idx": entry_idx,
        "exit_idx": exit_idx,
        "entry_price": entry_price,
        "sl_price": sl_price,
        "tp_price": tp_price,
        "exit_price": exit_price,
        "risk": risk,
        "r_multiple": r_multiple,
        "label": label,
        "exit_reason": exit_reason,
        "holding_bars": int(exit_idx - entry_idx),
    }


In [ ]:

FEATURE_COLUMNS = [
    "range", "body", "body_abs", "upper_wick", "lower_wick", "body_ratio", "close_position",
    "ret_1", "ret_3", "ret_5", "ret_10", "ret_20",
    "range_mean_5", "range_mean_10", "range_mean_20",
    "vol_std_5", "vol_std_10", "vol_std_20",
    "tr", "atr_14", "atr_50", "compression_ratio_20",
    "dist_high_10", "dist_low_10", "dist_high_20", "dist_low_20", "dist_high_50", "dist_low_50",
    "ema_20", "ema_50", "ema_slope_20", "ema_slope_50", "dist_ema_20", "dist_ema_50",
    "bb_mid", "bb_upper", "bb_lower", "bb_width", "bb_width_atr",
    "streak", "directional_eff_10", "inside_bar", "inside_cluster",
    "hour", "session",
]

def collect_trades_and_dataset(df: pd.DataFrame, cfg: ResearchConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    trades = []
    rows = []
    for event_name, (col, direction) in EVENT_MAP.items():
        signal_positions = np.flatnonzero(df[col].fillna(False).values)
        for pos in signal_positions:
            trade = simulate_trade(df, int(pos), direction, event_name, cfg)
            if trade is None:
                continue
            feature_row = df.iloc[pos][FEATURE_COLUMNS].to_dict()
            feature_row.update(trade)
            rows.append(feature_row)
            trades.append(trade)
    return pd.DataFrame(trades), pd.DataFrame(rows)


## 5. Metrics, split, dan mining breakdown

In [ ]:

def max_drawdown_r(r: pd.Series) -> float:
    eq = r.fillna(0).cumsum()
    dd = eq - eq.cummax()
    return float(dd.min()) if len(dd) else 0.0

def max_losing_streak(r: pd.Series) -> int:
    streak = 0
    best = 0
    for x in r.fillna(0):
        if x < 0:
            streak += 1
            best = max(best, streak)
        else:
            streak = 0
    return best

def compute_metrics(trades: pd.DataFrame) -> Dict[str, float]:
    if trades.empty:
        return {
            "total_trades": 0, "win_rate": np.nan, "profit_factor": np.nan, "expectancy_r": np.nan,
            "avg_r": np.nan, "median_r": np.nan, "sum_r": 0.0, "max_drawdown_r": 0.0,
            "max_losing_streak": 0, "avg_holding_bars": np.nan,
        }
    r = trades["r_multiple"]
    gross_profit = r[r > 0].sum()
    gross_loss = -r[r < 0].sum()
    pf = np.inf if gross_loss == 0 and gross_profit > 0 else (gross_profit / gross_loss if gross_loss > 0 else np.nan)
    return {
        "total_trades": int(len(trades)),
        "win_rate": float((r > 0).mean()),
        "profit_factor": float(pf),
        "expectancy_r": float(r.mean()),
        "avg_r": float(r.mean()),
        "median_r": float(r.median()),
        "sum_r": float(r.sum()),
        "max_drawdown_r": float(max_drawdown_r(r)),
        "max_losing_streak": int(max_losing_streak(r)),
        "avg_holding_bars": float(trades["holding_bars"].mean()),
    }

def split_by_time(df: pd.DataFrame, cfg: ResearchConfig) -> Dict[str, Tuple[pd.Timestamp, pd.Timestamp]]:
    n = len(df)
    i1 = int(n * cfg.split_train)
    i2 = int(n * (cfg.split_train + cfg.split_val))
    return {
        "train": (df.index[0], df.index[i1 - 1]),
        "val": (df.index[i1], df.index[i2 - 1]),
        "test": (df.index[i2], df.index[-1]),
    }

def assign_split(ts: pd.Series, split_ranges: Dict[str, Tuple[pd.Timestamp, pd.Timestamp]]) -> pd.Series:
    out = pd.Series(index=ts.index, dtype="object")
    tsv = pd.to_datetime(ts, utc=True)
    for name, (start, end) in split_ranges.items():
        out[(tsv >= start) & (tsv <= end)] = name
    return out

def breakdown_table(trades: pd.DataFrame, by: str) -> pd.DataFrame:
    rows = []
    for key, grp in trades.groupby(by):
        m = compute_metrics(grp)
        m[by] = key
        rows.append(m)
    if not rows:
        return pd.DataFrame()
    cols = [by, "total_trades", "win_rate", "profit_factor", "expectancy_r", "sum_r", "max_drawdown_r", "avg_holding_bars"]
    return pd.DataFrame(rows)[cols].sort_values(by)


## 6. Meta-labeling baseline

In [ ]:

def prepare_ml_data(dataset: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, List[str], List[str]]:
    ml = dataset.copy()
    ml["entry_ts"] = pd.to_datetime(ml["entry_ts"], utc=True)
    numeric_features = [c for c in FEATURE_COLUMNS if c != "session"]
    categorical_features = ["session", "event_name"]
    X = ml[numeric_features + categorical_features]
    y = ml["label"].astype(int)
    return X, y, numeric_features, categorical_features

def fit_meta_models(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> Dict[str, object]:
    X_train, y_train, _, cat_cols = prepare_ml_data(train_df)
    X_val, y_val, _, _ = prepare_ml_data(val_df)
    X_test, y_test, _, _ = prepare_ml_data(test_df)

    X_train_enc = pd.get_dummies(X_train, columns=cat_cols, dummy_na=True)
    X_val_enc = pd.get_dummies(X_val, columns=cat_cols, dummy_na=True)
    X_test_enc = pd.get_dummies(X_test, columns=cat_cols, dummy_na=True)
    X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)
    X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

    scaler = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    X_train_num = scaler.fit_transform(X_train_enc)
    X_val_num = scaler.transform(X_val_enc)
    X_test_num = scaler.transform(X_test_enc)

    logit = LogisticRegression(max_iter=1000, random_state=CFG.random_state)
    logit.fit(X_train_num, y_train)

    gbdt = HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=300, random_state=CFG.random_state)
    gbdt.fit(X_train_enc.fillna(0), y_train)

    return {
        "logit": {
            "model": logit,
            "val_proba": logit.predict_proba(X_val_num)[:, 1],
            "test_proba": logit.predict_proba(X_test_num)[:, 1],
            "y_val": y_val.values,
            "y_test": y_test.values,
        },
        "gbdt": {
            "model": gbdt,
            "val_proba": gbdt.predict_proba(X_val_enc.fillna(0))[:, 1],
            "test_proba": gbdt.predict_proba(X_test_enc.fillna(0))[:, 1],
            "y_val": y_val.values,
            "y_test": y_test.values,
        }
    }

def choose_threshold(y_true: np.ndarray, proba: np.ndarray, min_threshold: float = 0.50, max_threshold: float = 0.80) -> Tuple[float, pd.DataFrame]:
    thresholds = np.arange(min_threshold, max_threshold + 0.001, 0.01)
    rows = []
    for th in thresholds:
        pred = (proba >= th).astype(int)
        sel = pred.sum()
        precision = (y_true[pred == 1].mean() if sel > 0 else np.nan)
        rows.append({"threshold": round(float(th), 2), "selected": int(sel), "precision": precision})
    table = pd.DataFrame(rows).sort_values(["precision", "selected"], ascending=[False, False])
    best = float(table.iloc[0]["threshold"]) if not table.empty else 0.55
    return best, table


## 7. Pipeline eksekusi

In [ ]:

def run_research(cfg: ResearchConfig) -> Dict[str, object]:
    print("Loading data ...")
    df_raw = load_ohlcv(cfg.csv_path)
    report = quality_report(df_raw)
    print_quality_report(report)

    print("Engineering features ...")
    df = engineer_features(df_raw)
    df = build_events(df, cfg)

    print("Collecting trades and dataset ...")
    trades, dataset = collect_trades_and_dataset(df, cfg)
    if trades.empty or dataset.empty:
        raise ValueError("Tidak ada trades/dataset yang terbentuk. Review definisi event atau kualitas data.")

    split_ranges = split_by_time(df, cfg)
    trades["entry_ts"] = pd.to_datetime(trades["entry_ts"], utc=True)
    dataset["entry_ts"] = pd.to_datetime(dataset["entry_ts"], utc=True)
    trades["split"] = assign_split(trades["entry_ts"], split_ranges)
    dataset["split"] = assign_split(dataset["entry_ts"], split_ranges)

    metrics_all = compute_metrics(trades)
    metrics_by_split = {k: compute_metrics(v) for k, v in trades.groupby("split")}
    yearly = breakdown_table(trades.assign(year=trades["entry_ts"].dt.year), by="year")
    by_event = breakdown_table(trades, by="event_name")

    merged_session = dataset[["entry_ts", "event_name", "session", "r_multiple"]].copy()
    by_session = breakdown_table(merged_session, by="session")

    train_df = dataset[dataset["split"] == "train"].reset_index(drop=True)
    val_df = dataset[dataset["split"] == "val"].reset_index(drop=True)
    test_df = dataset[dataset["split"] == "test"].reset_index(drop=True)

    ml_results = fit_meta_models(train_df, val_df, test_df)
    threshold_summary = {}
    filtered_trade_outputs = {}

    for model_name, res in ml_results.items():
        best_th, table = choose_threshold(res["y_val"], res["val_proba"])
        threshold_summary[model_name] = {
            "best_threshold_from_val": best_th,
            "val_auc": float(roc_auc_score(res["y_val"], res["val_proba"])) if len(np.unique(res["y_val"])) > 1 else np.nan,
            "test_auc": float(roc_auc_score(res["y_test"], res["test_proba"])) if len(np.unique(res["y_test"])) > 1 else np.nan,
            "threshold_table": table,
        }

        test_scored = test_df.copy()
        test_scored["proba"] = res["test_proba"]
        selected = test_scored[test_scored["proba"] >= best_th].copy()
        selected_metrics = compute_metrics(selected)
        filtered_trade_outputs[model_name] = {
            "selected_test_df": selected,
            "selected_test_metrics": selected_metrics,
        }

    OUT_DIR = Path(cfg.out_dir)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    trades.to_csv(OUT_DIR / "trades.csv", index=False)
    dataset.to_csv(OUT_DIR / "meta_label_dataset.csv", index=False)

    metrics_payload = {
        "config": asdict(cfg),
        "quality_report": report,
        "overall": metrics_all,
        "by_split": metrics_by_split,
        "ml_threshold_summary": {
            k: {
                "best_threshold_from_val": v["best_threshold_from_val"],
                "val_auc": v["val_auc"],
                "test_auc": v["test_auc"],
            } for k, v in threshold_summary.items()
        },
        "filtered_test_metrics": {
            k: v["selected_test_metrics"] for k, v in filtered_trade_outputs.items()
        }
    }
    with open(OUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics_payload, f, indent=2, default=str)
    with open(OUT_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
        json.dump({
            "config": asdict(cfg),
            "feature_count": len(FEATURE_COLUMNS),
            "events": list(EVENT_MAP.keys()),
            "output_dir": str(OUT_DIR.resolve()),
        }, f, indent=2)

    return {
        "df": df,
        "trades": trades,
        "dataset": dataset,
        "metrics_all": metrics_all,
        "metrics_by_split": metrics_by_split,
        "yearly": yearly,
        "by_session": by_session,
        "by_event": by_event,
        "ml_results": ml_results,
        "threshold_summary": threshold_summary,
        "filtered_trade_outputs": filtered_trade_outputs,
        "split_ranges": split_ranges,
    }


## 8. Visualisasi

In [ ]:

def plot_equity_curve(trades: pd.DataFrame, title: str = "Equity Curve (R)") -> None:
    if trades.empty:
        print("No trades to plot.")
        return
    t = trades.sort_values("entry_ts").copy()
    t["cum_r"] = t["r_multiple"].cumsum()
    plt.figure(figsize=(12, 5))
    plt.plot(pd.to_datetime(t["entry_ts"]), t["cum_r"])
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel("Cumulative R")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_monthly_r(trades: pd.DataFrame, title: str = "Monthly Net R") -> pd.DataFrame:
    if trades.empty:
        print("No trades to plot.")
        return pd.DataFrame()
    t = trades.copy()
    t["entry_ts"] = pd.to_datetime(t["entry_ts"], utc=True)
    monthly = t.set_index("entry_ts")["r_multiple"].resample("M").sum().to_frame("net_r")
    plt.figure(figsize=(12, 5))
    plt.bar(monthly.index.astype(str), monthly["net_r"])
    plt.xticks(rotation=90)
    plt.title(title)
    plt.xlabel("Month")
    plt.ylabel("Net R")
    plt.tight_layout()
    plt.show()
    monthly.to_csv(Path(CFG.out_dir) / "monthly_R.csv")
    return monthly



## 9. Jalankan notebook

1. Ubah `CFG.csv_path`
2. Jalankan sel di bawah
3. Review output `trades.csv`, `metrics.json`, `run_manifest.json`, dan `meta_label_dataset.csv`


In [ ]:

# results = run_research(CFG)

# print("Overall metrics:")
# print(json.dumps(results["metrics_all"], indent=2))

# print("\nMetrics by split:")
# print(json.dumps(results["metrics_by_split"], indent=2))

# display(results["by_event"].head(20))
# display(results["yearly"])
# display(results["by_session"])

# plot_equity_curve(results["trades"])
# monthly_df = plot_monthly_r(results["trades"])

# for model_name, obj in results["filtered_trade_outputs"].items():
#     print(f"\nFiltered test metrics - {model_name}")
#     print(json.dumps(obj["selected_test_metrics"], indent=2))



## 10. Catatan pengembangan lanjutan

Notebook ini bisa diperluas ke:

- walk-forward loop per tahun / per quarter
- Monte Carlo reshuffle
- feature importance dan SHAP
- XGBoost / LightGBM jika tersedia
- event definitions tambahan berbasis BBMA OA
- parity mapping ke MT5 / MQL5
